# Week 2 — AI Logo & Design Studio (CNN + Colour Extraction)
**Tools:** TensorFlow/Keras, OpenCV, scikit-learn (KMeans)
**Dataset:** Logo Dataset (Kaggle) — 167,140 images, 2,341 classes

In [ ]:
!pip install tensorflow opencv-python-headless scikit-learn matplotlib pillow tqdm -q
import os,cv2,numpy as np,matplotlib.pyplot as plt,json,joblib
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras import layers,models,callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
IMG_SIZE,BATCH,EPOCHS=128,32,20
print('TF:',tf.__version__)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
LOGO_PATH='/content/drive/MyDrive/BrandSphere/logo_dataset/'
def load_logos(base,img_size=128,max_per_class=50):
    imgs,labels,classes=[],[],[]
    base=Path(base); cls_dirs=sorted([d for d in base.iterdir() if d.is_dir()])[:100]
    classes=[d.name for d in cls_dirs]; c2i={c:i for i,c in enumerate(classes)}
    for cls_dir in cls_dirs:
        files=list(cls_dir.glob('*.jpg'))+list(cls_dir.glob('*.png'))
        for f in files[:max_per_class]:
            try:
                img=cv2.imread(str(f)); img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
                img=cv2.resize(img,(img_size,img_size)); imgs.append(img/255.0); labels.append(c2i[cls_dir.name])
            except: pass
    print(f'Loaded {len(imgs)} images, {len(classes)} classes')
    return np.array(imgs),np.array(labels),classes
X,y,classes=load_logos(LOGO_PATH)
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
print(f'Train:{X_tr.shape} Test:{X_te.shape}')

In [ ]:
# Data augmentation
datagen=ImageDataGenerator(rotation_range=15,width_shift_range=.1,height_shift_range=.1,horizontal_flip=True,brightness_range=[.8,1.2],zoom_range=.1)
datagen.fit(X_tr); print('Augmentation ready ✓')

In [ ]:
# CNN architecture: Conv2D → MaxPool → Dense → Softmax
def build_cnn(num_classes,img_size=128):
    m=models.Sequential([
        layers.Conv2D(32,(3,3),activation='relu',padding='same',input_shape=(img_size,img_size,3)),
        layers.BatchNormalization(), layers.MaxPooling2D(2,2), layers.Dropout(.25),
        layers.Conv2D(64,(3,3),activation='relu',padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D(2,2), layers.Dropout(.25),
        layers.Conv2D(128,(3,3),activation='relu',padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D(2,2), layers.Dropout(.3),
        layers.Flatten(), layers.Dense(256,activation='relu'), layers.Dropout(.5),
        layers.Dense(num_classes,activation='softmax')
    ])
    m.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return m
model=build_cnn(len(classes)); model.summary()

In [ ]:
cb=[callbacks.EarlyStopping(patience=5,restore_best_weights=True),callbacks.ReduceLROnPlateau(factor=.5,patience=3),callbacks.ModelCheckpoint('best_logo_cnn.h5',save_best_only=True)]
history=model.fit(datagen.flow(X_tr,y_tr,batch_size=BATCH),steps_per_epoch=len(X_tr)//BATCH,validation_data=(X_te,y_te),epochs=EPOCHS,callbacks=cb)
loss,acc=model.evaluate(X_te,y_te); print(f'Test accuracy: {acc:.4f}')

In [ ]:
# Training curves
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4))
a1.plot(history.history['accuracy'],label='Train'); a1.plot(history.history['val_accuracy'],label='Val'); a1.set_title('Accuracy'); a1.legend()
a2.plot(history.history['loss'],label='Train'); a2.plot(history.history['val_loss'],label='Val'); a2.set_title('Loss'); a2.legend()
plt.tight_layout(); plt.savefig('week2_training.png',dpi=150); plt.show()

In [ ]:
# Extract embeddings from Dense-256 layer
embed_model=models.Model(inputs=model.input,outputs=model.layers[-3].output)
embeds=embed_model.predict(X_te); np.save('logo_embeddings.npy',embeds); np.save('logo_labels.npy',y_te)
# PCA visualisation
pca=PCA(n_components=2); red=pca.fit_transform(embeds)
plt.figure(figsize=(10,8)); plt.scatter(red[:,0],red[:,1],c=y_te,cmap='tab10',s=15,alpha=.7)
plt.title('Logo Embedding Clusters (PCA 2D)',fontweight='bold'); plt.colorbar(label='Class')
plt.savefig('week2_pca.png',dpi=150); plt.show(); print('Embeddings saved ✓')

In [ ]:
# KMeans colour palette extraction
def extract_palette(img_arr,n=5):
    pix=(img_arr*255).reshape(-1,3).astype(np.float32)
    km=KMeans(n_clusters=n,random_state=42,n_init='auto').fit(pix)
    cols=km.cluster_centers_.astype(int)
    prop=np.bincount(km.labels_)/len(km.labels_)
    order=np.argsort(-prop); cols=cols[order]
    return ['#{:02x}{:02x}{:02x}'.format(*c) for c in cols],cols
hex_pal,rgb_pal=extract_palette(X_te[0])
fig,axes=plt.subplots(1,6,figsize=(12,2))
axes[0].imshow(X_te[0]); axes[0].set_title('Logo',fontsize=9); axes[0].axis('off')
for i,(ax,c) in enumerate(zip(axes[1:],rgb_pal)): ax.imshow([[c/255]]); ax.set_title(hex_pal[i],fontsize=7); ax.axis('off')
plt.suptitle('Extracted Colour Palette',fontweight='bold'); plt.tight_layout(); plt.savefig('week2_palette.png',dpi=150); plt.show()

## ✅ Week 2 Deliverables
- [ ] Logo images loaded from Kaggle dataset (class-folder structure)
- [ ] Preprocessing: resize 128×128, normalise 0–1, augmentation
- [ ] CNN trained: Conv2D→MaxPool→BatchNorm×3→Dense256→Softmax
- [ ] Embeddings extracted (256-dim) and saved
- [ ] PCA cluster visualisation saved
- [ ] KMeans colour palette extraction function ready